<a href="https://colab.research.google.com/github/sinbbang/L-IntelligentSystem/blob/main/%EC%A7%80%EB%8A%A5%EC%8B%9C%EC%8A%A4%ED%85%9C_07-2_Routing_Workflow_Gemini_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 실습 2. Routing도 Workflow다

**목표:** 사용자 질문을 `refund / technical / faq` 중 하나로 분류하고, **개발자가 미리 지정한 세 경로 중 하나만** 실행합니다.

- **기본 실습(무료·키 불필요):** 키워드 기반 분류 + 3개 실제 Python 처리 함수
- **심화 실습(선택):** **Google Gemini 3.5 Flash-Lite**가 질문을 분류하도록 변경
- **핵심:** LLM이 분류 라벨을 정해도 가능한 경로는 개발자가 고정했으므로 *Routing Workflow*입니다.

> 주의: 아래 서비스는 실제 환불·이메일·계정 작업을 수행하지 않는 **교육용 시뮬레이션**입니다. API 사용량과 무료 한도는 계정별로 달라질 수 있습니다.

## 1. 실행 준비
Colab에서 **런타임 → 모두 실행**을 선택하세요. 선택 실습을 수행할 때만 Google AI Studio의 Gemini API 키가 필요합니다.

In [1]:
# 표준 라이브러리만 사용합니다.
from datetime import datetime

# 개발자가 허용하는 경로를 미리 정의합니다.
ROUTES = {
    "refund": "refund_workflow",
    "technical": "tech_workflow",
    "faq": "faq_workflow",
}
print("사용 가능한 경로:", ROUTES)

사용 가능한 경로: {'refund': 'refund_workflow', 'technical': 'tech_workflow', 'faq': 'faq_workflow'}


## 2. 가장 간단한 Router: 분류 결과를 경로에 매핑

In [2]:
def route(category):
    # 분류 결과가 사전에 정의된 세 경로 중 어느 것인지 확인합니다.
    # 예기치 못한 라벨이면 faq_workflow로 보냅니다(교육용 기본값).
    return ROUTES.get(category, "faq_workflow")

for category in ["refund", "technical", "faq", "unknown"]:
    print(category, "→", route(category))

refund → refund_workflow
technical → tech_workflow
faq → faq_workflow
unknown → faq_workflow


## 3. 입력 문장을 키워드로 분류

In [3]:
def classify_by_rules(question):
    # 실제 서비스에서는 더 정교한 분류 규칙이나 학습 모델을 사용할 수 있습니다.
    text = question.lower()
    if any(word in text for word in ["환불", "취소", "반품", "refund"]):
        return "refund"
    if any(word in text for word in ["로그인", "오류", "에러", "고장", "접속", "technical"]):
        return "technical"
    return "faq"

questions = [
    "결제를 취소하고 환불받고 싶어요.",
    "로그인이 안 되고 오류가 떠요.",
    "서비스 이용 시간은 언제인가요?",
]
for q in questions:
    print(f"{q} → {classify_by_rules(q)}")

결제를 취소하고 환불받고 싶어요. → refund
로그인이 안 되고 오류가 떠요. → technical
서비스 이용 시간은 언제인가요? → faq


## 4. 세 개의 독립적인 처리 경로 구현
함수는 실제 환불이나 설정 변경을 하지 않고 안내 문구만 반환합니다.

In [4]:
def refund_workflow(question):
    # 환불 관련 요청만 처리하는 고정된 단계입니다.
    return "[환불 안내] 주문번호와 결제일을 확인한 뒤 환불 정책을 안내합니다."

def tech_workflow(question):
    # 기술 지원 요청만 처리합니다.
    return "[기술 지원] 오류 메시지와 사용 환경을 확인하고 재현 절차를 안내합니다."

def faq_workflow(question):
    # 일반 정보 관련 요청을 처리합니다.
    return "[일반 FAQ] 서비스 이용 방법과 자주 묻는 질문을 안내합니다."

HANDLERS = {
    "refund": refund_workflow,
    "technical": tech_workflow,
    "faq": faq_workflow,
}

## 5. 전체 Routing Workflow 실행 및 로그 확인

In [5]:
def run_routing_workflow(question, classifier=classify_by_rules):
    logs = []  # 실행 단계를 기록할 리스트
    category = classifier(question)  # 1. 분류 (규칙 또는 LLM)
    logs.append(("classify", category))

    # 2. 모델의 라벨이 세 경로 중 하나인지 확인합니다.
    # 운영 환경에서는 모호한 분류를 사용자에게 재확인할 수도 있습니다.
    if category not in HANDLERS:
        category = "faq"
    logs.append(("route", ROUTES[category]))

    # 3. 미리 정의된 함수 중 하나만 실행합니다.
    answer = HANDLERS[category](question)
    logs.append(("execute", category))
    return {"question": question, "category": category, "answer": answer, "logs": logs}

for question in questions:
    result = run_routing_workflow(question)
    print("\n질문:", result["question"])
    print("결과:", result["answer"])
    print("로그:", result["logs"])


질문: 결제를 취소하고 환불받고 싶어요.
결과: [환불 안내] 주문번호와 결제일을 확인한 뒤 환불 정책을 안내합니다.
로그: [('classify', 'refund'), ('route', 'refund_workflow'), ('execute', 'refund')]

질문: 로그인이 안 되고 오류가 떠요.
결과: [기술 지원] 오류 메시지와 사용 환경을 확인하고 재현 절차를 안내합니다.
로그: [('classify', 'technical'), ('route', 'tech_workflow'), ('execute', 'technical')]

질문: 서비스 이용 시간은 언제인가요?
결과: [일반 FAQ] 서비스 이용 방법과 자주 묻는 질문을 안내합니다.
로그: [('classify', 'faq'), ('route', 'faq_workflow'), ('execute', 'faq')]


## 6. 자동 확인: Routing은 후보 경로를 늘리지 않는다

In [6]:
for q in questions:
    result = run_routing_workflow(q)
    assert result["category"] in HANDLERS
    assert [step for step, _ in result["logs"]] == ["classify", "route", "execute"]
print("✓ 세 경로 중 하나만 실행되고, 단계 순서가 고정되어 있습니다.")

✓ 세 경로 중 하나만 실행되고, 단계 순서가 고정되어 있습니다.


## 7. 선택 심화: 실제 무료 티어 Gemini로 분류하기

1. [Google AI Studio](https://aistudio.google.com/apikey)에서 API 키를 발급받습니다.
2. Colab 왼쪽 **🔑 Secrets**에서 `GEMINI_API_KEY`를 등록하고 노트북 액세스를 허용합니다.
3. 아래 셀을 실행하여 실제 질문을 분류합니다.

사용 모델 ID: `gemini-3.5-flash-lite` (공식 문서: https://ai.google.dev/gemini-api/docs/models/gemini-3.5-flash-lite). **무료 사용 여부 및 제한량은 실행 시 계정에서 확인**하세요. API 키를 코드에 직접 적거나 제출하지 마세요.

In [11]:
%pip -q install -U google-genai

In [15]:
from getpass import getpass
from google import genai
import requests

# 입력한 키는 코드 셀에 표시되지 않습니다.
GEMINI_API_KEY = getpass("Gemini API key: ").strip()

client = genai.Client(api_key=GEMINI_API_KEY)
MODEL_ID = "gemini-3.5-flash-lite"
print("설정 완료 (API 키 값은 출력하지 않습니다).")

Gemini API key: ··········
설정 완료 (API 키 값은 출력하지 않습니다).


In [16]:
def classify_with_gemini(question):
    # 핵심: LLM에게 새 워크플로우 설계 권한을 주지 않습니다.
    # 세 가지 라벨 중 하나만 선택하도록 요청합니다.
    if client is None:
        raise RuntimeError("Colab Secrets에 GEMINI_API_KEY를 설정하세요.")
    prompt = (
        "사용자 문의를 다음 셋 중 딱 하나로 분류하세요: "
        "refund(환불/취소), technical(오류/접속), faq(기타). "
        "설명 없이 라벨 하나만 출력하세요.\n"
        f"사용자 문의: {question}"
    )
    response = client.models.generate_content(
        model=MODEL_ID, contents=prompt,
        config={"temperature": 0, "max_output_tokens": 30},
    )
    label = (response.text or "").strip().lower()
    # 모델이 약속과 다른 문자열을 반환해도 허용 범위를 넘지 않도록 검증합니다.
    if label not in HANDLERS:
        raise ValueError(f"예상하지 못한 분류 라벨: {label!r}")
    return label

if client is not None:
    try:
        q = "수강 등록 후 결제를 취소하고 싶습니다."
        result = run_routing_workflow(q, classifier=classify_with_gemini)
        print("LLM 분류:", result["category"])
        print("실행 결과:", result["answer"])
        print("실행 로그:", result["logs"])
    except Exception as exc:
        print("API 호출에 실패했습니다. 키·모델 접근 권한·할당량을 확인하세요.")
        print(type(exc).__name__, str(exc)[:300])
else:
    print("실제 API 실습을 건너뜁니다. 위 셀에서 키를 설정하면 실행할 수 있습니다.")

LLM 분류: refund
실행 결과: [환불 안내] 주문번호와 결제일을 확인한 뒤 환불 정책을 안내합니다.
실행 로그: [('classify', 'refund'), ('route', 'refund_workflow'), ('execute', 'refund')]


## 8. 학생 실습 문제

1. `"배송이 늦어요"`를 분류해 보세요. 예상한 경로와 결과가 같은가요?
2. `"환불하려는데 로그인이 안 돼요"`처럼 **두 범주가 동시에 나타나는 문장**을 입력하여 규칙 기반과 Gemini 분류를 비교하세요.
3. `"배송"`이라는 새 범주를 추가하려면 **개발자가** `ROUTES`와 `HANDLERS`에 무엇을 추가해야 할까요?
4. 새로운 분류 모델을 붙여도 `classify → route → execute`의 **전체 실행 구조가 고정**되어 있는지 로그로 설명하세요.

In [17]:
# 직접 입력 문장을 바꾸어 확인하세요.
my_question = "환불받고 싶지만 로그인이 되지 않아요."
my_result = run_routing_workflow(my_question)
print("분류:", my_result["category"])
print("처리:", my_result["answer"])
print("로그:", my_result["logs"])

분류: refund
처리: [환불 안내] 주문번호와 결제일을 확인한 뒤 환불 정책을 안내합니다.
로그: [('classify', 'refund'), ('route', 'refund_workflow'), ('execute', 'refund')]


## 9. 정리

- LLM이 선택하는 것은 `refund / technical / faq` **분류 라벨**입니다.
- 개발자가 고정한 것은 **가능한 경로, 처리 함수, 실행 순서**입니다.
- 즉, LLM이 라우팅을 결정한다고 해서 자동으로 Agent가 되는 것은 아닙니다.

**다음 실습 연결:** Agent에서는 중간 결과를 보고 새로운 행동을 선택하거나 이전 단계를 다시 수행할 수 있습니다.